# Long-only ETF portfolio construction options

这个 notebook 只看 training sample：2010-01-01 到 2020-12-31。`TRAIN_END_EXCLUSIVE` 设为 2021-01-01 只是为了让 Yahoo Finance 下载时包含 2020 年最后一个交易日；后续会显式裁剪并检查没有 2021+ 数据。

目标：在 long-only、fully invested（权重非负、权重和为 1）的约束下，比较几种 ETF portfolio construction 方案：

1. **Equal weight**：稳健 baseline，不估计参数。
2. **Inverse volatility**：低波 ETF 权重大，控制组合波动。
3. **12-1 momentum top-k**：只持有过去 12 个月、跳过最近 1 个月后表现最好的 ETF。
4. **Momentum + inverse volatility**：先选强趋势 ETF，再按低波倾斜。
5. **Rolling mean-variance**：用短期 trailing mean/cov 做 long-only MV。
6. **Active mean-variance vs SPY**：目标是相对 SPY 的 active utility，适合问“能不能比 SPY 更好”。

所有策略信号都在 rebalance date 当天只使用历史窗口，`run_backtest` 会把权重 lag 1 个交易日，避免当日信号当日成交的 look-ahead。


In [ ]:
import pandas as pd

from alpha_lab.analytics.returns import cumulative_returns
from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.active_mv import rolling_active_mean_variance_weights
from alpha_lab.portfolio.long_only import (
    rolling_equal_weight_weights,
    rolling_inverse_volatility_weights,
    rolling_mean_variance_weights,
    rolling_momentum_weights,
)
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly


In [ ]:
TRAIN_START = "2010-01-01"
TRAIN_END = "2020-12-31"
TRAIN_END_EXCLUSIVE = "2021-01-01"

BENCHMARK = "SPY"
REBALANCE_MONTHLY = "ME"
REBALANCE_WEEKLY = "W-FRI"

LOOKBACK_3M = 63
LOOKBACK_12M = 252
SKIP_1M = 21
TOP_N = 4
MV_RISK_AVERSION = 5.0
ACTIVE_RISK_AVERSION = 1.0

COSTS_BPS = 1.0
SLIPPAGE_BPS = 2.0


## ETF universe

用 liquid、历史较长、覆盖多资产类别的 ETF：美股、海外股票、债券、信用、商品、黄金、REITs。这样 long-only 组合不只是 sector rotation，也能在 risk-off 环境里切到 duration/gold。


In [ ]:
universe = pd.DataFrame(
    [
        ("SPY", "US equity"),
        ("EFA", "Developed ex-US equity"),
        ("EEM", "Emerging markets equity"),
        ("IEF", "7-10Y Treasury"),
        ("TLT", "20Y+ Treasury"),
        ("LQD", "Investment grade credit"),
        ("HYG", "High yield credit"),
        ("GLD", "Gold"),
        ("DBC", "Broad commodities"),
        ("VNQ", "US REITs"),
    ],
    columns=["ticker", "asset_class"],
)
TICKERS = universe["ticker"].tolist()
universe


In [ ]:
raw_prices = load_prices(TICKERS, start=TRAIN_START, end=TRAIN_END_EXCLUSIVE)
prices = raw_prices.loc[TRAIN_START:TRAIN_END].dropna(how="all")

# Keep only ETFs with a complete enough training history. This avoids accidental late-inception assets.
min_obs = int(0.90 * len(prices))
valid_tickers = prices.count()[prices.count() >= min_obs].index.tolist()
prices = prices[valid_tickers].ffill().dropna(how="any")

benchmark_prices = prices[BENCHMARK]
active_asset_prices = prices.drop(columns=[BENCHMARK])

pd.Series(
    {
        "first_price_date": prices.index.min().date(),
        "last_price_date": prices.index.max().date(),
        "n_price_rows": len(prices),
        "n_assets": prices.shape[1],
        "has_after_2020": bool((prices.index > TRAIN_END).any()),
    }
)


In [ ]:
assert prices.index.min() >= pd.Timestamp(TRAIN_START)
assert prices.index.max() <= pd.Timestamp(TRAIN_END)
assert not (prices.index > TRAIN_END).any()


## Construct long-only strategy weights

Tradeoff note: the first four methods are deliberately robust and low-parameter; the MV methods can adapt faster, but are more estimation-error sensitive.


In [ ]:
strategy_weights = {
    "01_equal_weight_monthly": rolling_equal_weight_weights(
        prices,
        rebalance=REBALANCE_MONTHLY,
    ),
    "02_inverse_vol_3m_monthly": rolling_inverse_volatility_weights(
        prices,
        lookback_days=LOOKBACK_3M,
        rebalance=REBALANCE_MONTHLY,
    ),
    "03_momentum_12_1_top4_monthly": rolling_momentum_weights(
        prices,
        lookback_days=LOOKBACK_12M,
        skip_days=SKIP_1M,
        top_n=TOP_N,
        rebalance=REBALANCE_MONTHLY,
        vol_adjust=False,
    ),
    "04_momentum_12_1_top4_invvol_monthly": rolling_momentum_weights(
        prices,
        lookback_days=LOOKBACK_12M,
        skip_days=SKIP_1M,
        top_n=TOP_N,
        rebalance=REBALANCE_MONTHLY,
        vol_adjust=True,
        vol_lookback_days=LOOKBACK_3M,
    ),
    "05_mean_variance_3m_weekly": rolling_mean_variance_weights(
        prices,
        lookback_days=LOOKBACK_3M,
        rebalance=REBALANCE_WEEKLY,
        risk_aversion=MV_RISK_AVERSION,
    ),
    "06_active_mv_vs_spy_3m_weekly": rolling_active_mean_variance_weights(
        active_asset_prices,
        benchmark_prices,
        lookback_days=LOOKBACK_3M,
        rebalance=REBALANCE_WEEKLY,
        risk_aversion=ACTIVE_RISK_AVERSION,
    ),
}

# Reindex active MV back to the full price matrix; it intentionally excludes SPY as a candidate asset.
strategy_weights["06_active_mv_vs_spy_3m_weekly"] = strategy_weights[
    "06_active_mv_vs_spy_3m_weekly"
].reindex(columns=prices.columns, fill_value=0.0)

weight_checks = pd.DataFrame(
    {
        name: {
            "first_rebalance": weights.index.min(),
            "last_rebalance": weights.index.max(),
            "min_weight": weights.min().min(),
            "max_row_sum_error": (weights.sum(axis=1) - 1).abs().max(),
        }
        for name, weights in strategy_weights.items()
    }
).T
weight_checks


## Backtest only inside training sample

The benchmark is SPY buy-and-hold over the same realized dates after each strategy becomes invested. Costs are simple one-way bps assumptions, applied to turnover.


In [ ]:
def evaluate_strategy(name: str, weights: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    result = run_backtest(
        weights,
        prices,
        costs_bps=COSTS_BPS,
        slippage_bps=SLIPPAGE_BPS,
    )
    first_invested = result.weights.abs().sum(axis=1).gt(0).idxmax()
    strategy_returns = result.returns.loc[first_invested:].rename(name)
    benchmark_returns = benchmark_prices.pct_change().reindex(strategy_returns.index).fillna(0.0)
    benchmark_returns = benchmark_returns.rename("SPY buy&hold")
    active_returns = (strategy_returns - benchmark_returns).rename(name)
    return strategy_returns, benchmark_returns, active_returns

strategy_returns = {}
active_returns = {}
benchmark_by_strategy = {}
for name, weights in strategy_weights.items():
    strategy_r, benchmark_r, active_r = evaluate_strategy(name, weights)
    strategy_returns[name] = strategy_r
    active_returns[name] = active_r
    benchmark_by_strategy[name] = benchmark_r

returns = pd.DataFrame(strategy_returns).dropna(how="all")
active = pd.DataFrame(active_returns).dropna(how="all")
returns.tail()


In [ ]:
summary_table = pd.DataFrame({name: pd.Series(summary(series)) for name, series in returns.items()}).T

active_table = pd.DataFrame(
    {
        name: {
            "ActiveReturn": active_series.mean() * 252,
            "TrackingError": active_series.std() * (252**0.5),
            "InformationRatio": active_series.mean() / active_series.std() * (252**0.5),
            "ActiveHitRate": (active_series > 0).mean(),
            "FinalRelativeWealth": cumulative_returns(strategy_returns[name]).iloc[-1]
            / cumulative_returns(benchmark_by_strategy[name]).iloc[-1],
        }
        for name, active_series in active_returns.items()
    }
).T

comparison = summary_table.join(active_table).sort_values(
    ["InformationRatio", "Sharpe"],
    ascending=False,
)
comparison.style.format("{:.2%}", subset=["CAGR", "AnnVol", "MaxDD", "HitRate", "ActiveReturn", "TrackingError", "ActiveHitRate"]).format("{:.2f}")


In [ ]:
best_by_ir = comparison.index[0]
best_by_sharpe = comparison["Sharpe"].idxmax()
pd.Series({"best_by_information_ratio": best_by_ir, "best_by_sharpe": best_by_sharpe})


## Visual checks

Focus on: final wealth, relative wealth vs SPY, active drawdown, and whether the strategy only wins in one short subperiod.


In [ ]:
equity_curve(returns, name="long-only ETF strategy wealth, 2010-2020 training only")


In [ ]:
spy_for_panel = benchmark_prices.pct_change().reindex(returns.index).fillna(0.0).rename("SPY buy&hold")
equity_curve(pd.concat([returns, spy_for_panel], axis=1), name="strategies vs SPY")


In [ ]:
relative_wealth = pd.DataFrame(
    {
        name: cumulative_returns(strategy_returns[name]) / cumulative_returns(benchmark_by_strategy[name])
        for name in strategy_returns
    }
)
relative_wealth.plot(title="Relative wealth vs SPY, by strategy")


In [ ]:
active_drawdown = relative_wealth / relative_wealth.cummax() - 1
active_drawdown.plot(title="Active drawdown vs SPY")


In [ ]:
drawdown_chart(returns[best_by_ir])


In [ ]:
heatmap_monthly(active[best_by_ir])


## Inspect latest training-sample allocations

这些权重是 2020 年内最后几个 rebalance 的结果；不要用 2021+ 数据评价或调参。


In [ ]:
strategy_weights[best_by_ir].tail(12).style.format("{:.1%}")


In [ ]:
avg_weights = pd.DataFrame({name: weights.mean() for name, weights in strategy_weights.items()}).T
avg_weights.style.format("{:.1%}")
